In [1]:
%%writefile app.py
import os
import json
import datetime
import streamlit as st
from pathlib import Path
from dotenv import load_dotenv
from tavily import TavilyClient
from google import genai
from llama_index.core import Document

st.set_page_config(page_title="🦙 LlamaIndex Research Agent", layout="wide")

# Force explicit path to .env file relative to working directory
env_path = Path('.env').resolve()
load_dotenv(dotenv_path=env_path, override=True)

gemini_key = os.getenv("GEMINI_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

st.sidebar.title("⚙️ Agent Settings")
if gemini_key and tavily_key:
    st.sidebar.success("✅ API Keys Loaded")
else:
    st.sidebar.error("❌ API Keys Missing in .env")
    st.sidebar.write(f"Looking at path: `{env_path}`")

@st.cache_resource
def get_clients(g_key, t_key):
    return genai.Client(api_key=g_key), TavilyClient(api_key=t_key)

st.title("🦙 LlamaIndex Autonomous Research Agent")

topic = st.text_input("Research Topic / Query:", placeholder="e.g., PAK VS SL TEST SERIES 2026")
start_btn = st.button("🚀 Run LlamaIndex Agent", type="primary")

if start_btn:
    if not topic.strip():
        st.warning("Please enter a research topic.")
    elif not gemini_key or not tavily_key:
        st.error("API keys missing from .env file.")
    else:
        try:
            gemini_client, tavily_client = get_clients(gemini_key, tavily_key)

            with st.status("🧠 Decomposing Topic with Gemini...", expanded=True) as status:
                planner_prompt = f'Break down "{topic}" into 3 specific web search queries. Return ONLY a valid JSON array of 3 strings.'
                planner_res = gemini_client.models.generate_content(model="gemini-3.5-flash-lite", contents=planner_prompt)
                sub_queries = json.loads(planner_res.text.strip().replace("```json", "").replace("```", ""))
                status.update(label="✅ Query Decomposition Complete", state="complete")

            with st.status("🌐 Building LlamaIndex Documents...", expanded=True) as status:
                all_documents = []
                for idx, q in enumerate(sub_queries, 1):
                    s_res = tavily_client.search(query=q, max_results=2, search_depth="basic")
                    for res in s_res.get("results", []):
                        all_documents.append(Document(text=res.get("content", ""), extra_info={"url": res.get("url", ""), "sub_query": q}))
                status.update(label="✅ Ingestion Complete", state="complete")

            with st.status("🤖 Synthesizing Final Report...", expanded=True) as status:
                combined_context = "\n\n---\n\n".join([f"Query: {doc.extra_info.get('sub_query')}\nSource: {doc.extra_info.get('url')}\nContent: {doc.text}" for doc in all_documents])
                writer_prompt = f'Synthesize this into a research report on "{topic}":\n\n{combined_context}'
                final_res = gemini_client.models.generate_content(model="gemini-3.5-flash-lite", contents=writer_prompt)
                status.update(label="✅ Synthesis Complete", state="complete")

            report_text = final_res.text
            filename = f"llamaindex_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(report_text)

            st.success(f"💾 Saved as: `{filename}`")
            st.markdown(report_text)
            st.download_button("📥 Download Report (.md)", report_text, file_name=filename)

        except Exception as e:
            st.error(f"❌ Error: {str(e)}")

Overwriting app.py


In [5]:
import os
import json
import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
from dotenv import load_dotenv
from tavily import TavilyClient
from google import genai
from llama_index.core import Document

# 1. Load Environment Variables
load_dotenv(override=True)

gemini_key = os.getenv("GEMINI_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

# 2. Key Check Display
if gemini_key and tavily_key:
    status_html = "<h4 style='color:green;'>✅ Both API Keys Loaded (Gemini & Tavily)</h4>"
elif gemini_key:
    status_html = "<h4 style='color:orange;'>⚠️ Only Gemini Key found. Tavily Key missing in .env!</h4>"
else:
    status_html = "<h4 style='color:red;'>❌ API Keys missing in .env file!</h4>"

# 3. Widgets UI Layout
header = widgets.HTML("<h2>🦙 Native LlamaIndex Deep Research Agent</h2>")
status_widget = widgets.HTML(status_html)

topic_input = widgets.Text(
    placeholder="e.g., Quantum Computing Breakthroughs 2026",
    description="Topic:",
    layout=widgets.Layout(width='60%')
)

run_button = widgets.Button(
    description="Start Research Agent",
    button_style='success',
    icon='rocket'
)

output_area = widgets.Output()

# 4. Agent Execution Routine
def run_agent_pipeline(b):
    with output_area:
        output_area.clear_output()
        topic = topic_input.value.strip()

        if not topic:
            print("⚠️ Topic enter karna zaroori hai.")
            return

        g_key = os.getenv("GEMINI_API_KEY")
        t_key = os.getenv("TAVILY_API_KEY")

        if not g_key or not t_key:
            print("❌ Error: API Keys missing. Dono GEMINI_API_KEY aur TAVILY_API_KEY aapki .env file mein hone chahiye.")
            return

        try:
            gemini_client = genai.Client(api_key=g_key)
            tavily_client = TavilyClient(api_key=t_key)

            # STEP 1: Query Decomposition
            print(f"🧠 [Step 1/3] Decomposing topic via Gemini: '{topic}'...")
            planner_prompt = f"""
Break down the topic "{topic}" into 3 specific, targeted web search queries.
Return ONLY a valid JSON array of 3 strings. Example: ["query 1", "query 2", "query 3"]
"""
            planner_res = gemini_client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=planner_prompt
            )
            
            clean_json = planner_res.text.strip().replace("```json", "").replace("```", "")
            sub_queries = json.loads(clean_json)

            print(f"🎯 Sub-queries generated:\n 1. {sub_queries[0]}\n 2. {sub_queries[1]}\n 3. {sub_queries[2]}\n")

            # STEP 2: Fetch Web Data & Construct LlamaIndex Document Nodes
            print("🌐 [Step 2/3] Fetching Web Context & Building LlamaIndex Documents...")
            all_documents = []

            for idx, q in enumerate(sub_queries, 1):
                print(f" -> Searching [{idx}/3]: {q}")
                s_res = tavily_client.search(query=q, max_results=2, search_depth="basic")
                results = s_res.get("results", [])

                for res in results:
                    doc = Document(
                        text=res.get("content", ""),
                        extra_info={
                            "url": res.get("url", ""),
                            "sub_query": q
                        }
                    )
                    all_documents.append(doc)

            print(f"✅ Created {len(all_documents)} LlamaIndex Document nodes.\n")

            # Format LlamaIndex Documents for Synthesis
            combined_context = "\n\n---\n\n".join([
                f"Query: {doc.extra_info.get('sub_query')}\nSource: {doc.extra_info.get('url')}\nContent: {doc.text}" 
                for doc in all_documents
            ])

            # STEP 3: Synthesis
            print("🤖 [Step 3/3] Synthesizing final research report with Gemini 3.5 Flash-Lite...")
            writer_prompt = f"""
You are an expert AI Research Analyst using LlamaIndex document pipelines.
Synthesize the retrieved document context below into a comprehensive research report on: "{topic}".

Requirements:
- Executive Summary
- Key Technical Insights (Categorized by sub-topic)
- Architectural & Future Implications
- References (List URLs)

Retrieved LlamaIndex Context:
{combined_context}
"""
            final_res = gemini_client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=writer_prompt
            )

            # Save File
            filename = f"llamaindex_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(f"# LlamaIndex Deep Research Report: {topic}\n\n{final_res.text}")

            output_area.clear_output()
            print(f"💾 Report saved to file: {filename}\n")
            display(Markdown(f"# LlamaIndex Deep Research Report: {topic}"))
            display(Markdown(final_res.text))

        except Exception as e:
            print(f"❌ Error occurred: {str(e)}")

run_button.on_click(run_agent_pipeline)

# 5. Display Dashboard Interface
display(header)
display(status_widget)
display(widgets.HBox([topic_input, run_button]))
display(widgets.HTML("<hr>"))
display(output_area)

HTML(value='<h2>🦙 Native LlamaIndex Deep Research Agent</h2>')

HTML(value="<h4 style='color:green;'>✅ Both API Keys Loaded (Gemini & Tavily)</h4>")

HTML(value='<hr>')

Output()

In [ ]:
!streamlit run app.py